In [ ]:
# =========================================================
# STEP 1 — Setup workspace (Kaggle only, no Colab)
# =========================================================
# Mục tiêu notebook này:
# 1) Train LoRA cho 3 domain: POSTER, LOGO_2D, LOGO_3D
# 2) Làm sạch caption để tránh lỗi token quá dài / loss NaN
# 3) Xuất artifact gọn nhẹ để tải về local và dùng lại trong backend

REPO_URL = "https://github.com/My-Name-Hung/AI_GEN_IMGAE.git"
PROJECT_DIR = "/kaggle/working/AI_GEN"

!rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

print("✅ Project ready:", PROJECT_DIR)

In [ ]:
# =========================================================
# STEP 2 — Install dependencies (DO NOT upgrade torch here)
# =========================================================
%cd /kaggle/working/AI_GEN

# Quan trọng:
# - Không cài torch trong cell này, để tránh ghi đè bản tương thích GPU.
# - Torch stack sẽ được xử lý ở STEP 2.1.

!pip -q install --upgrade pip
!pip -q install \
  "diffusers>=0.31.0" "transformers==4.41.2" "accelerate==0.34.2" \
  "peft==0.12.0" "safetensors>=0.4.3" "huggingface_hub>=0.23.0" \
  "open_clip_torch>=2.23.0" "Pillow>=10.0.0" "opencv-python>=4.8.0" \
  "opencv-contrib-python>=4.8.0" "scikit-image>=0.22.0" "scipy>=1.11.0" \
  "fastapi>=0.109.0" "uvicorn[standard]>=0.25.0" "python-multipart>=0.0.6" \
  "aiofiles>=22.1.0" "pydantic>=2.5.0" "python-dotenv>=1.0.0" \
  "numpy>=1.24.0" "tqdm>=4.66.0"

print("✅ Base dependencies installed")

In [ ]:
# =========================================================
# STEP 3 — Configure Kaggle dataset paths + training profiles
# =========================================================
# Cập nhật DATASET_ROOT theo đúng Kaggle Input.
# Ví dụ: /kaggle/input/gen-ai-data

import os
from pathlib import Path

DATASET_ROOT = Path('/kaggle/input/gen-ai-data')  # <-- EDIT nếu cần
WORK_DIR = Path('/kaggle/working')
PREPARED_ROOT = WORK_DIR / 'prepared_lora_dataset'
OUTPUT_ROOT = WORK_DIR / 'outputs'

# 3 domain cần train
DOMAINS = {
    'poster': {
        'raw_dir': DATASET_ROOT / 'POSTER',
        'token': 'poster_style',
        'val_prompt': 'poster_style, modern cinematic poster, strong typography, high detail'
    },
    'logo_2d': {
        'raw_dir': DATASET_ROOT / 'LOGO' / 'LOGO 2D',
        'token': 'logo2d_style',
        'val_prompt': 'logo2d_style, clean vector-like logo, minimal, centered, white background'
    },
    'logo_3d': {
        'raw_dir': DATASET_ROOT / 'LOGO' / 'LOGO 3D',
        'token': 'logo3d_style',
        'val_prompt': 'logo3d_style, glossy 3d logo mockup, soft studio light, centered'
    },
}

for k, cfg in DOMAINS.items():
    if not cfg['raw_dir'].exists():
        raise FileNotFoundError(f"❌ Missing dataset folder for {k}: {cfg['raw_dir']}")

PREPARED_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('✅ Dataset root:', DATASET_ROOT)
print('✅ Output root :', OUTPUT_ROOT)
print('✅ Domains     :', ', '.join(DOMAINS.keys()))

In [ ]:
# =========================================================
# STEP 4 — Prepare data for all domains (clean + paired image/text)
# =========================================================
# bước này sẽ:
# - chuẩn hóa caption
# - cắt caption theo token budget an toàn
# - loại caption rác (ví dụ "-", lặp số)
# - resize ảnh về 512 để train ổn định trên T4

import re
import shutil
from PIL import Image
from pathlib import Path

MAX_CHARS = 220  # giữ caption ngắn để giảm nguy cơ truncate quá mạnh
TARGET_SIZE = 512
VALID_EXT = {'.png', '.jpg', '.jpeg', '.webp', '.bmp'}


def clean_caption(text: str, token: str) -> str:
    t = (text or '').strip().lower()
    t = t.replace('\n', ' ')
    t = re.sub(r'\s+', ' ', t)

    # loại caption vô nghĩa thường gây NaN / học sai
    bad_patterns = [
        r'^[-_\s]+$',
        r'^(the\s*){3,}$',
        r'^(\d\s*){6,}$',
        r'^(spa\s*){2,}$',
    ]
    if any(re.search(p, t) for p in bad_patterns):
        t = ''

    if len(t) < 8:
        t = f'{token}, clean design, high quality'

    # ép prefix token để domain rõ ràng
    if token not in t:
        t = f'{token}, {t}'

    # cắt độ dài để tránh vượt tokenizer 77 tokens quá nhiều
    t = t[:MAX_CHARS].strip(' ,')
    return t


def prepare_domain(domain_name: str, raw_dir: Path, token: str, out_root: Path):
    out_dir = out_root / domain_name
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    count = 0
    skipped = 0

    for img_path in sorted(raw_dir.iterdir()):
        if not img_path.is_file() or img_path.suffix.lower() not in VALID_EXT:
            continue

        txt_path = img_path.with_suffix('.txt')
        if not txt_path.exists():
            skipped += 1
            continue

        try:
            img = Image.open(img_path).convert('RGB').resize((TARGET_SIZE, TARGET_SIZE), Image.BICUBIC)
            base = f'{domain_name}_{count:05d}'
            out_img = out_dir / f'{base}.png'
            out_txt = out_dir / f'{base}.txt'

            raw_caption = txt_path.read_text(encoding='utf-8', errors='ignore')
            caption = clean_caption(raw_caption, token)

            img.save(out_img)
            out_txt.write_text(caption, encoding='utf-8')
            count += 1
        except Exception:
            skipped += 1

    print(f'[{domain_name}] prepared={count}, skipped={skipped}, dir={out_dir}')
    return out_dir


prepared_dirs = {}
for name, cfg in DOMAINS.items():
    prepared_dirs[name] = prepare_domain(
        domain_name=name,
        raw_dir=cfg['raw_dir'],
        token=cfg['token'],
        out_root=PREPARED_ROOT,
    )

print('✅ Data preparation done for all domains')

In [15]:
# =========================================================
# STEP 5 — Train 3 LoRA models sequentially on Kaggle (best-practice)
# =========================================================
# NOTE quan trọng:
# - Không train chung poster + logo vào 1 LoRA (dễ nhiễu style)
# - Train 3 LoRA riêng: poster, logo_2d, logo_3d
# - Mỗi domain có token riêng để prompt rõ ràng

import os
import sys
import subprocess
from pathlib import Path

os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')
os.environ.setdefault('TORCH_COMPILE_DISABLE', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

PROJECT_DIR = Path('/kaggle/working/AI_GEN')
os.chdir(PROJECT_DIR)

# Hotfix dtype để tránh lỗi float/half mismatch trên Kaggle
train_file = PROJECT_DIR / 'training/lora_trainer/train_lora.py'
if train_file.exists():
    src = train_file.read_text(encoding='utf-8')
    old = 'pixel_values = batch["pixel_values"].to(device)'
    new = (
        'vae_dtype = next(vae.parameters()).dtype\n'
        '            pixel_values = batch["pixel_values"].to(device=device, dtype=vae_dtype)'
    )
    if old in src and new not in src:
        src = src.replace(old, new, 1)
        train_file.write_text(src, encoding='utf-8')
        print('✅ Applied Kaggle hotfix for VAE dtype')

# Hyperparams an toàn cho T4 (ưu tiên ổn định hơn tốc độ)
BASE_MODEL = 'stabilityai/sdxl-turbo'
COMMON_ARGS = {
    'rank': '8',
    'alpha': '16',
    'lr': '5e-5',
    'epochs': '6',
    'batch_size': '1',
    'grad_accum': '8',
    'resolution': '512',
    'save_steps': '100',
}


def train_one_domain(domain_name: str, dataset_dir: Path, val_prompt: str):
    out_dir = OUTPUT_ROOT / f'lora_{domain_name}'

    cmd = [
        sys.executable, '-m', 'training.lora_trainer.train_lora',
        '--model_name', BASE_MODEL,
        '--dataset', str(dataset_dir),
        '--output', str(out_dir),
        '--rank', COMMON_ARGS['rank'],
        '--alpha', COMMON_ARGS['alpha'],
        '--lr', COMMON_ARGS['lr'],
        '--epochs', COMMON_ARGS['epochs'],
        '--batch_size', COMMON_ARGS['batch_size'],
        '--grad_accum', COMMON_ARGS['grad_accum'],
        '--resolution', COMMON_ARGS['resolution'],
        '--save_steps', COMMON_ARGS['save_steps'],
        '--validation_prompt', val_prompt,
    ]

    print('\n' + '=' * 70)
    print(f'🚀 Start training domain: {domain_name}')
    print('Dataset:', dataset_dir)
    print('Output :', out_dir)
    print('=' * 70)

    r = subprocess.run(cmd, env=os.environ.copy())
    if r.returncode != 0:
        raise RuntimeError(f'Training failed for {domain_name} (exit={r.returncode})')

    print(f'✅ Completed: {domain_name}')


for domain, cfg in DOMAINS.items():
    train_one_domain(
        domain_name=domain,
        dataset_dir=prepared_dirs[domain],
        val_prompt=cfg['val_prompt'],
    )

print('\n🎉 All 3 LoRA trainings completed')
print('Poster :', OUTPUT_ROOT / 'lora_poster')
print('Logo2D :', OUTPUT_ROOT / 'lora_logo_2d')
print('Logo3D :', OUTPUT_ROOT / 'lora_logo_3d')


Applied Kaggle hotfix: cast pixel_values to VAE dtype
Starting LoRA training...
Dataset: /kaggle/working/dataset_processed
Output : /kaggle/working/outputs/lora_poster
First download may take 5-15 min. Do NOT press Stop.


2026-04-06 04:36:31.652967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775450191.675631     946 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775450191.683190     946 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775450191.702870     946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775450191.702918     946 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775450191.702924     946 computation_placer.cc:177] computation placer alr

trainable params: 11,612,160 || all params: 2,579,075,844 || trainable%: 0.4502


/kaggle/working/AI_GEN/training/lora_trainer/train_lora.py:248: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_ctx():
Epoch 1/10:   3%|▎         | 18/590 [00:28<14:01,  1.47s/it, loss=nan]Token indices sequence length is longer than the specified maximum sequence length for this model (87 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['exhibition of the exhibition of the exhibition of the exhibition']
Token indices sequence length is longer than the specified maximum sequence length for this model (87 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['exhibition of the exhibition of the exhibition of the exhibition']
Epoch 1/10:   6%|▋   

Training complete!


In [ ]:
# =========================================================
# STEP 6 — Quick inference test for 3 trained LoRA models
# =========================================================
import os
import torch
from pathlib import Path
from diffusers import AutoPipelineForText2Image

base_model = 'stabilityai/sdxl-turbo'
pipeline = AutoPipelineForText2Image.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    use_safetensors=True,
).to('cuda')

tests = [
    {
        'name': 'poster',
        'lora_dir': OUTPUT_ROOT / 'lora_poster' / 'final' / 'unet_lora',
        'prompt': 'poster_style, cinematic movie poster, dramatic light, clean composition'
    },
    {
        'name': 'logo_2d',
        'lora_dir': OUTPUT_ROOT / 'lora_logo_2d' / 'final' / 'unet_lora',
        'prompt': 'logo2d_style, minimal geometric logo, flat design, white background'
    },
    {
        'name': 'logo_3d',
        'lora_dir': OUTPUT_ROOT / 'lora_logo_3d' / 'final' / 'unet_lora',
        'prompt': 'logo3d_style, glossy 3d emblem, soft shadow, premium branding'
    },
]

test_out = OUTPUT_ROOT / 'tests'
test_out.mkdir(parents=True, exist_ok=True)

for item in tests:
    lora_dir = item['lora_dir']
    if not lora_dir.exists():
        print(f"⚠️ Skip {item['name']} (missing {lora_dir})")
        continue

    pipeline.unload_lora_weights()
    pipeline.load_lora_weights(str(lora_dir))

    image = pipeline(
        item['prompt'],
        num_inference_steps=4,
        guidance_scale=1.0,
    ).images[0]

    out_path = test_out / f"test_{item['name']}.png"
    image.save(out_path)
    print(f"✅ Saved: {out_path}")


2026-04-06 07:11:10.851757: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775459470.874707      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775459470.882185      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775459470.902344      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775459470.902365      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775459470.902368      55 computation_placer.cc:177] computation placer alr

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

text_encoder_2/model.safetensors:   0%|          | 0.00/2.78G [00:00<?, ?B/s]

text_encoder/model.safetensors:   0%|          | 0.00/492M [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/10.3G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

Saved to /kaggle/working/outputs/test_output.png


In [17]:
# =========================================================
# STEP 7 — Verify final adapters for all domains
# =========================================================
!ls -lh /kaggle/working/outputs/lora_poster/final/unet_lora/
!ls -lh /kaggle/working/outputs/lora_logo_2d/final/unet_lora/
!ls -lh /kaggle/working/outputs/lora_logo_3d/final/unet_lora/

total 45M
-rw-r--r-- 1 root root  758 Apr  6 07:07 adapter_config.json
-rw-r--r-- 1 root root  45M Apr  6 07:07 adapter_model.safetensors
-rw-r--r-- 1 root root 5.1K Apr  6 07:07 README.md


In [18]:
# =========================================================
# STEP 8 — Package all LoRA outputs to ZIP (download from Kaggle Output)
# =========================================================
!cd /kaggle/working/outputs && zip -r lora_poster.zip lora_poster
!cd /kaggle/working/outputs && zip -r lora_logo_2d.zip lora_logo_2d
!cd /kaggle/working/outputs && zip -r lora_logo_3d.zip lora_logo_3d

print('✅ ZIP ready:')
print('/kaggle/working/outputs/lora_poster.zip')
print('/kaggle/working/outputs/lora_logo_2d.zip')
print('/kaggle/working/outputs/lora_logo_3d.zip')

# Gợi ý dùng local sau khi tải về:
# - giải nén vào: outputs/lora_poster, outputs/lora_logo_2d, outputs/lora_logo_3d
# - backend generate: truyền lora_path đến .../final/unet_lora
# - hoặc set DEFAULT_LORA_PATH theo domain bạn muốn mặc định

  adding: lora_poster/ (stored 0%)
  adding: lora_poster/checkpoint-400/ (stored 0%)
  adding: lora_poster/checkpoint-400/unet_lora/ (stored 0%)
  adding: lora_poster/checkpoint-400/unet_lora/adapter_config.json (deflated 53%)
  adding: lora_poster/checkpoint-400/unet_lora/README.md (deflated 65%)
  adding: lora_poster/checkpoint-400/unet_lora/adapter_model.safetensors (deflated 68%)
  adding: lora_poster/checkpoint-100/ (stored 0%)
  adding: lora_poster/checkpoint-100/unet_lora/ (stored 0%)
  adding: lora_poster/checkpoint-100/unet_lora/adapter_config.json (deflated 53%)
  adding: lora_poster/checkpoint-100/unet_lora/README.md (deflated 65%)
  adding: lora_poster/checkpoint-100/unet_lora/adapter_model.safetensors (deflated 68%)
  adding: lora_poster/val_epoch_2.png (deflated 0%)
  adding: lora_poster/val_epoch_8.png (deflated 0%)
  adding: lora_poster/val_epoch_10.png (deflated 0%)
  adding: lora_poster/checkpoint-200/ (stored 0%)
  adding: lora_poster/checkpoint-200/unet_lora/ (store